In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nkongolo/ugransome-dataset/LIDETA.ipynb
/kaggle/input/datasets/nkongolo/ugransome-dataset/final(2).csv


In [2]:
import glob, time, random, warnings, json, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); random.seed(SEED)

df = pd.read_csv(glob.glob('/kaggle/input/**/*.csv', recursive=True)[0])
df = df.drop_duplicates().dropna().reset_index(drop=True)

TARGET = 'Prediction'
y = LabelEncoder().fit_transform(df[TARGET])
X = df.drop(columns=[TARGET]).copy()
for c in X.select_dtypes(include='object').columns:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))

feature_names = X.columns.tolist()
n_feat = len(feature_names)
X = X.values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED)

# Subset used only inside GA fitness
GA_N = min(10000, len(X_train))
idx = np.random.RandomState(SEED).choice(len(X_train), GA_N, replace=False)
Xga_tr, Xga_val, yga_tr, yga_val = train_test_split(
    X_train[idx], y_train[idx], test_size=0.3, stratify=y_train[idx], random_state=SEED)

# Baseline
rf0 = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED).fit(X_train, y_train)
p = rf0.predict(X_test)
base_acc, base_f1 = accuracy_score(y_test, p), f1_score(y_test, p, average='macro')
print(f"Baseline RF acc={base_acc:.4f} macroF1={base_f1:.4f}")

Baseline RF acc=0.9932 macroF1=0.9932


In [3]:
N_EST = [50, 100, 150, 200]
DEPTH = [None, 5, 10, 15, 20, 30]
MSS   = [2, 5, 10]
MAXF  = ['sqrt', 'log2', 0.5]
HP_SIZES = [len(N_EST), len(DEPTH), len(MSS), len(MAXF)]

def repair(ind):
    if ind[:n_feat].sum() == 0:
        ind[np.random.randint(n_feat)] = 1
    return ind

def random_ind():
    mask = np.random.randint(0, 2, n_feat)
    hp = np.array([np.random.randint(s) for s in HP_SIZES])
    return repair(np.concatenate([mask, hp]))

def build_rf(ind):
    hp = ind[n_feat:]
    return RandomForestClassifier(
        n_estimators=N_EST[hp[0]], max_depth=DEPTH[hp[1]],
        min_samples_split=MSS[hp[2]], max_features=MAXF[hp[3]],
        n_jobs=-1, random_state=SEED)

cache = {}
def evaluate(ind):
    key = tuple(ind)
    if key not in cache:
        cols = np.where(ind[:n_feat] == 1)[0]
        rf = build_rf(ind).fit(Xga_tr[:, cols], yga_tr)
        cache[key] = (accuracy_score(yga_val, rf.predict(Xga_val[:, cols])), len(cols))
    return cache[key]

def fitness(ind):                      # single-objective
    acc, k = evaluate(ind)
    return acc - 0.01 * (k / n_feat)

def objs(ind):                         # multi-objective (minimise error, feature fraction)
    acc, k = evaluate(ind)
    return np.array([1 - acc, k / n_feat])

def tourn_idx(fit, k=3):
    i = np.random.randint(len(fit), size=k)
    return i[np.argmax(fit[i])]

def crossover(a, b):
    m = np.random.rand(len(a)) < 0.5
    return np.where(m, a, b), np.where(m, b, a)

def mutate(ind, pm):
    ind = ind.copy()
    for i in range(n_feat):
        if np.random.rand() < pm: ind[i] = 1 - ind[i]
    for j, s in enumerate(HP_SIZES):
        if np.random.rand() < pm * 3: ind[n_feat + j] = np.random.randint(s)
    return repair(ind)

POP, GENS = 16, 10
results = {}

In [4]:
def sga(pop_size=POP, gens=GENS, pc=0.8, pm=0.1, elite=1):
    pop = [random_ind() for _ in range(pop_size)]
    for g in range(gens):
        fit = np.array([fitness(i) for i in pop])
        new = [pop[i].copy() for i in np.argsort(-fit)[:elite]]
        while len(new) < pop_size:
            p1, p2 = pop[tourn_idx(fit)], pop[tourn_idx(fit)]
            c1, c2 = crossover(p1, p2) if np.random.rand() < pc else (p1.copy(), p2.copy())
            new += [mutate(c1, pm), mutate(c2, pm)]
        pop = new[:pop_size]
    fit = np.array([fitness(i) for i in pop])
    return pop[int(np.argmax(fit))]

def aga(pop_size=POP, gens=GENS, k1=0.9, k2=0.1, k3=0.9, k4=0.1, elite=1):
    pop = [random_ind() for _ in range(pop_size)]
    for g in range(gens):
        fit = np.array([fitness(i) for i in pop])
        fmax, favg = fit.max(), fit.mean()
        den = max(fmax - favg, 1e-9)
        new = [pop[i].copy() for i in np.argsort(-fit)[:elite]]
        while len(new) < pop_size:
            i1, i2 = tourn_idx(fit), tourn_idx(fit)
            fp = max(fit[i1], fit[i2])
            pc = k1 * (fmax - fp) / den if fp >= favg else k3
            pm = k2 * (fmax - fp) / den if fp >= favg else k4
            if np.random.rand() < pc: c1, c2 = crossover(pop[i1], pop[i2])
            else: c1, c2 = pop[i1].copy(), pop[i2].copy()
            new += [mutate(c1, pm), mutate(c2, pm)]
        pop = new[:pop_size]
    fit = np.array([fitness(i) for i in pop])
    return pop[int(np.argmax(fit))]

def local_search(ind, iters=6):
    best, bf = ind.copy(), fitness(ind)
    for _ in range(iters):
        cand = best.copy()
        if np.random.rand() < 0.6:
            j = np.random.randint(n_feat); cand[j] = 1 - cand[j]
        else:
            j = np.random.randint(len(HP_SIZES))
            cand[n_feat + j] = np.random.randint(HP_SIZES[j])
        cand = repair(cand); cf = fitness(cand)
        if cf > bf: best, bf = cand, cf
    return best

def memetic(pop_size=POP, gens=GENS, pc=0.8, pm=0.1, ls_top=3):
    pop = [random_ind() for _ in range(pop_size)]
    for g in range(gens):
        fit = np.array([fitness(i) for i in pop])
        for i in np.argsort(-fit)[:ls_top]:
            pop[i] = local_search(pop[i])          # Lamarckian
        fit = np.array([fitness(i) for i in pop])
        new = [pop[int(np.argmax(fit))].copy()]
        while len(new) < pop_size:
            p1, p2 = pop[tourn_idx(fit)], pop[tourn_idx(fit)]
            c1, c2 = crossover(p1, p2) if np.random.rand() < pc else (p1.copy(), p2.copy())
            new += [mutate(c1, pm), mutate(c2, pm)]
        pop = new[:pop_size]
    fit = np.array([fitness(i) for i in pop])
    return pop[int(np.argmax(fit))]

for name, fn in [('SGA', sga), ('AGA', aga), ('Memetic', memetic)]:
    t = time.time(); ind = fn()
    results[name] = dict(ind=ind, time=time.time() - t)
    print(name, "done", evaluate(ind))

SGA done (0.982, 9)
AGA done (0.972, 6)
Memetic done (0.9846666666666667, 8)


In [5]:
def dominates(a, b):
    return np.all(a <= b) and np.any(a < b)

def fast_nd_sort(F):
    n = len(F); S = [[] for _ in range(n)]; cnt = np.zeros(n, int); fronts = [[]]
    for p in range(n):
        for q in range(n):
            if p == q: continue
            if dominates(F[p], F[q]): S[p].append(q)
            elif dominates(F[q], F[p]): cnt[p] += 1
        if cnt[p] == 0: fronts[0].append(p)
    i = 0
    while fronts[i]:
        nxt = []
        for p in fronts[i]:
            for q in S[p]:
                cnt[q] -= 1
                if cnt[q] == 0: nxt.append(q)
        i += 1; fronts.append(nxt)
    return fronts[:-1]

def crowding(F, front):
    d = np.zeros(len(front))
    for m in range(F.shape[1]):
        order = np.argsort(F[front, m])
        d[order[0]] = d[order[-1]] = np.inf
        rng = F[front[order[-1]], m] - F[front[order[0]], m]
        if rng == 0: continue
        for k in range(1, len(front) - 1):
            d[order[k]] += (F[front[order[k+1]], m] - F[front[order[k-1]], m]) / rng
    return d

def best_of_front(front):
    return max(front, key=lambda i: (evaluate(i)[0], -evaluate(i)[1]))

def nsga2(pop_size=POP, gens=GENS, pc=0.9, pm=0.1):
    pop = [random_ind() for _ in range(pop_size)]
    for g in range(gens):
        F = np.array([objs(i) for i in pop])
        rank = np.zeros(len(pop), int); crowd = np.zeros(len(pop))
        for r, fr in enumerate(fast_nd_sort(F)):
            rank[fr] = r; crowd[fr] = crowding(F, fr)
        def pick():
            a, b = np.random.randint(len(pop), size=2)
            if rank[a] != rank[b]: return a if rank[a] < rank[b] else b
            return a if crowd[a] > crowd[b] else b
        kids = []
        while len(kids) < pop_size:
            p1, p2 = pop[pick()], pop[pick()]
            c1, c2 = crossover(p1, p2) if np.random.rand() < pc else (p1.copy(), p2.copy())
            kids += [mutate(c1, pm), mutate(c2, pm)]
        union = pop + kids[:pop_size]
        FU = np.array([objs(i) for i in union])
        new = []
        for fr in fast_nd_sort(FU):
            if len(new) + len(fr) <= pop_size: new += fr
            else:
                order = np.argsort(-crowding(FU, fr))
                new += [fr[i] for i in order[:pop_size - len(new)]]
                break
        pop = [union[i] for i in new]
    F = np.array([objs(i) for i in pop])
    return [pop[i] for i in fast_nd_sort(F)[0]]

def spea2_fitness(F):
    n = len(F)
    dom = np.array([[dominates(F[i], F[j]) for j in range(n)] for i in range(n)])
    S = dom.sum(1)
    R = np.array([S[dom[:, i]].sum() for i in range(n)])
    D = np.linalg.norm(F[:, None, :] - F[None, :, :], axis=2)
    k = max(1, int(np.sqrt(n)))
    sigma = np.sort(D, axis=1)[:, k]
    return R + 1.0 / (sigma + 2.0), D

def spea2_select(F, size):
    fit, D = spea2_fitness(F)
    sel = [int(i) for i in np.where(fit < 1)[0]]
    if len(sel) < size:
        sel += [int(i) for i in np.argsort(fit) if i not in sel][:size - len(sel)]
    while len(sel) > size:
        sub = D[np.ix_(sel, sel)].copy(); np.fill_diagonal(sub, np.inf)
        srt = np.sort(sub, axis=1)
        sel.pop(min(range(len(sel)), key=lambda i: tuple(srt[i])))
    return sel

def spea2(pop_size=POP, arch_size=POP, gens=GENS, pc=0.9, pm=0.1):
    pop = [random_ind() for _ in range(pop_size)]; archive = []
    for g in range(gens):
        union = pop + archive
        FU = np.array([objs(i) for i in union])
        archive = [union[i] for i in spea2_select(FU, arch_size)]
        fitA, _ = spea2_fitness(np.array([objs(i) for i in archive]))
        def pick():
            a, b = np.random.randint(len(archive), size=2)
            return a if fitA[a] < fitA[b] else b
        kids = []
        while len(kids) < pop_size:
            p1, p2 = archive[pick()], archive[pick()]
            c1, c2 = crossover(p1, p2) if np.random.rand() < pc else (p1.copy(), p2.copy())
            kids += [mutate(c1, pm), mutate(c2, pm)]
        pop = kids[:pop_size]
    union = pop + archive
    FU = np.array([objs(i) for i in union])
    archive = [union[i] for i in spea2_select(FU, arch_size)]
    FA = np.array([objs(i) for i in archive])
    return [archive[i] for i in fast_nd_sort(FA)[0]]

for name, fn in [('NSGA-II', nsga2), ('SPEA2', spea2)]:
    t = time.time(); pareto = fn()
    results[name] = dict(ind=best_of_front(pareto), pareto=pareto, time=time.time() - t)
    print(name, "Pareto (acc, #feat):", sorted({evaluate(i) for i in pareto}, reverse=True))

NSGA-II Pareto (acc, #feat): [(0.9833333333333333, 8), (0.982, 6), (0.9726666666666667, 5), (0.968, 4), (0.9596666666666667, 3), (0.942, 2), (0.813, 1)]
SPEA2 Pareto (acc, #feat): [(0.9823333333333333, 10), (0.9816666666666667, 7), (0.9726666666666667, 6), (0.971, 5), (0.9643333333333334, 4), (0.9116666666666666, 3)]


In [6]:
rows = [dict(Method='Baseline RF', Accuracy=base_acc, MacroF1=base_f1,
             Features=n_feat, GA_time_s=0)]
for name, r in results.items():
    ind = r['ind']; cols = np.where(ind[:n_feat] == 1)[0]
    pred = build_rf(ind).fit(X_train[:, cols], y_train).predict(X_test[:, cols])
    rows.append(dict(Method=name,
                     Accuracy=accuracy_score(y_test, pred),
                     MacroF1=f1_score(y_test, pred, average='macro'),
                     Features=len(cols), GA_time_s=round(r['time'], 1)))
    hp = ind[n_feat:]
    print(f"{name}: n_est={N_EST[hp[0]]}, depth={DEPTH[hp[1]]}, mss={MSS[hp[2]]}, "
          f"max_feat={MAXF[hp[3]]} | {[feature_names[i] for i in cols]}")

comp = pd.DataFrame(rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)
display(comp)

comp.to_csv('/kaggle/working/results_comparison.csv', index=False)
best_name = comp.iloc[0]['Method']
if best_name == 'Baseline RF':
    best_model, best_cols = rf0, list(range(n_feat))
else:
    ind = results[best_name]['ind']
    best_cols = np.where(ind[:n_feat] == 1)[0].tolist()
    best_model = build_rf(ind).fit(X_train[:, best_cols], y_train)

joblib.dump(best_model, '/kaggle/working/best_rf_model.pkl')
json.dump({'method': best_name, 'features': [feature_names[i] for i in best_cols]},
          open('/kaggle/working/best_features.json', 'w'))
print("Saved:", best_name)

SGA: n_est=200, depth=20, mss=2, max_feat=sqrt | ['Time', 'Protcol', 'Flag', 'Family', 'Clusters', 'BTC', 'USD', 'Netflow_Bytes', 'Threats']
AGA: n_est=100, depth=15, mss=5, max_feat=sqrt | ['Time', 'Family', 'Clusters', 'BTC', 'USD', 'Netflow_Bytes']
Memetic: n_est=200, depth=30, mss=2, max_feat=0.5 | ['Family', 'Clusters', 'SeddAddress', 'BTC', 'USD', 'Netflow_Bytes', 'IPaddress', 'Threats']
NSGA-II: n_est=100, depth=15, mss=5, max_feat=log2 | ['Protcol', 'Family', 'Clusters', 'BTC', 'USD', 'Netflow_Bytes', 'IPaddress', 'Threats']
SPEA2: n_est=150, depth=None, mss=2, max_feat=0.5 | ['Flag', 'Family', 'Clusters', 'ExpAddress', 'BTC', 'USD', 'Netflow_Bytes', 'IPaddress', 'Threats', 'Port']


,Method,Accuracy,MacroF1,Features,GA_time_s
0,Baseline RF,0.993246,0.993238,13,0.0
1,NSGA-II,0.991390,0.991372,8,92.1
2,SPEA2,0.990607,0.990665,10,100.7
3,Memetic,0.990361,0.990434,8,182.0
4,SGA,0.990137,0.990181,9,102.5
5,AGA,0.984121,0.983895,6,12.8


Saved: Baseline RF


In [7]:
!git --version

git version 2.43.0


In [9]:
!git config --global user.name "DiyaMalikk07"
